## Part 1 - Attention Mechanisms, Multi-Head Attention, Memory Optimization, and Positional Encodings

**Covered:** QKV mechanics, scaled dot-product attention, causal masking, MHA, KV cache analysis, MQA/GQA tradeoff, Flash Attention (tiling + online softmax), Sliding Window Attention, RoPE/ALiBi positional encodings, Pre-Norm, RMSNorm.

In [ ]:
# 1. The Transformer's Core Computation: The Attention Mechanism

## 1. The Transformer's Core Computation: The Attention Mechanism

### 1.1 Why Attention? The Problem With Sequential Processing

Before transformers, sequence modeling was dominated by RNNs and LSTMs. Their fundamental flaw: **hidden state is a fixed-size bottleneck**. No matter how long the input sequence, you compress everything into a single vector $h_t$ before generating the next token. This creates an information bottleneck that degrades over long sequences (the "vanishing gradient over time" problem, distinct from depth-based vanishing gradients).

The key insight of attention: **instead of compressing the past into a single vector, allow every output position to directly attend to every input position**. This makes the dependency path between any two tokens $O(1)$ regardless of distance, compared to $O(n)$ for RNNs.

---

### 1.2 The QKV Framework: Formal Derivation

Every token $x_i$ in the input sequence is first embedded into a $d_{model}$-dimensional vector. Within each attention layer, three separate linear projections transform this embedding:

$$Q = XW^Q, \quad K = XW^K, \quad V = XW^V$$

where:
- $X \in \mathbb{R}^{n \times d_{model}}$ is the input sequence matrix
- $W^Q, W^K \in \mathbb{R}^{d_{model} \times d_k}$ are the query/key projection matrices
- $W^V \in \mathbb{R}^{d_{model} \times d_v}$ is the value projection matrix

The **query** $q_i$ represents *what token $i$ is looking for* — a vector in a learned "question" subspace.

The **key** $k_j$ represents *what token $j$ can offer* — a vector in a learned "advertisement" subspace.

The **value** $v_j$ is *the actual content* to be retrieved if there's a match.

The attention output for token $i$ is:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Intuition on the dot product:** $q_i \cdot k_j = \|q_i\|\|k_j\|\cos\theta$. If $q$ and $k$ point in the same direction (high cosine similarity), you get a large positive score. Orthogonal vectors give ~0. The dot product is a similarity measure in high-dimensional space.

---

### 1.3 The Scaling Factor $\frac{1}{\sqrt{d_k}}$: Why It's Non-Negotiable

This is often treated as a minor implementation detail. It is not. It's foundational to training stability.

**The statistical argument:** Assume $q$ and $k$ are vectors of dimension $d_k$ with i.i.d. components drawn from $\mathcal{N}(0, 1)$. The dot product $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$ has:

$$\mathbb{E}[q \cdot k] = 0, \quad \text{Var}[q \cdot k] = d_k$$

So the standard deviation grows as $\sqrt{d_k}$. For $d_k = 64$, std $= 8$. For $d_k = 512$, std $= 22.6$. Without scaling, the raw dot products can be in the range $[-30, +30]$ easily.

**Why this destroys softmax:** The softmax function is:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

For large magnitude inputs, $e^{z_i}$ for the dominant element becomes astronomically large compared to others. The output distribution becomes a near-one-hot vector. Backpropagating through a near-one-hot softmax means the gradient $\frac{\partial \text{softmax}}{\partial z}$ is approximately zero everywhere — **the vanishing gradient problem caused by softmax saturation.**

Dividing by $\sqrt{d_k}$ normalizes the variance back to 1, keeping inputs to softmax in a regime where the distribution is smooth and gradients are non-zero.

---

### 1.4 Causal Masking (Decoder-Only)

In decoder-only models, attention must be **causal**: token $i$ cannot attend to token $j > i$. This is enforced by adding a mask before softmax:

$$\text{score}_{ij} = \begin{cases} q_i \cdot k_j / \sqrt{d_k} & \text{if } j \leq i \\ -\infty & \text{if } j > i \end{cases}$$

$-\infty$ becomes $0$ after softmax, so future tokens effectively don't exist. This is how **auto-regressive generation** is trained on full sequences in parallel — even though at inference time tokens are generated one at a time.

---

In [ ]:
# 2. Multi-Head Attention (MHA)

## 2. Multi-Head Attention (MHA)

### 2.1 Motivation: One Subspace Is Never Enough

A single attention head learns **one** linear projection of the relational structure. Language has multiple simultaneously relevant relationship types:
- **Syntactic:** subject-verb agreement, dependency parsing
- **Semantic:** word sense disambiguation (bank → river vs. bank → finance)
- **Coreference:** pronoun resolution across long spans
- **Positional/local:** n-gram patterns, idioms

Forcing all of this into a single $d_k$-dimensional space creates destructive interference. Multi-head attention is the solution.

### 2.2 Formal Specification

For $h$ heads, with $d_k = d_v = d_{model}/h$:

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)W^O$$

where $W^O \in \mathbb{R}^{hd_v \times d_{model}}$ is the output projection.

**Dimensionality note:** Each head operates in a $d_k = 128$ subspace (for $d_{model}=4096$, $h=32$). The heads run **truly in parallel** on the GPU as batched matrix multiplications — there is no sequential dependency between heads.

**Parameter count for one MHA layer:**
- $W^Q, W^K, W^V$: $3 \times d_{model} \times d_{model}$
- $W^O$: $d_{model} \times d_{model}$
- Total: $4 d_{model}^2$ (same as four square matrices of the model dimension)

For $d_{model} = 4096$: $4 \times 4096^2 \approx 67M$ parameters per attention layer.

---

In [ ]:
# 3. The KV Cache: The Production Bottleneck

## 3. The KV Cache: The Production Bottleneck

### 3.1 Why It Exists

During auto-regressive decoding (inference), to generate token $t+1$, the model must compute attention over all tokens $1..t$. Naively recomputing all $K, V$ projections at every step costs $O(t)$ per step and $O(t^2)$ total. The **KV cache** stores all previously computed $K$ and $V$ vectors, making each step $O(1)$ in compute (amortized), but $O(t)$ in memory.

### 3.2 Memory Cost Analysis

For a model with:
- $L$ layers
- $h$ attention heads
- $d_k$ head dimension
- Sequence length $n$
- Batch size $B$
- Precision $p$ bytes (2 for FP16/BF16)

$$\text{KV Cache Size} = 2 \times L \times h \times d_k \times n \times B \times p$$

For Llama-2 70B ($L=80$, $h=64$, $d_k=128$, $n=4096$, $B=1$, $p=2$):

$$= 2 \times 80 \times 64 \times 128 \times 4096 \times 1 \times 2 \approx 10.7 \text{ GB}$$

For $B=10$ concurrent users: **107 GB** — just for the KV cache, before even storing model weights (~140 GB in FP16). This is why MHA is economically brutal at scale.

---

In [ ]:
# 4. MQA → GQA: The Memory-Quality Tradeoff Spectrum

## 4. MQA → GQA: The Memory-Quality Tradeoff Spectrum

### 4.1 Multi-Query Attention (MQA)

**Shazeer, 2019.** Radical simplification: all $h$ query heads share **one** K head and **one** V head.

$$\text{head}_i = \text{Attention}(QW_i^Q, KW^K_{\text{shared}}, VW^V_{\text{shared}})$$

**Memory reduction:** KV cache scales by $1/h$ instead of $1$. For $h=64$: **64× smaller KV cache**.

**The cost:** Each query head asks a different question but looks at the same K/V representation. The model loses the ability to build diverse relational subspaces. Empirically: ~1-2% perplexity degradation on benchmarks, but **4-5× inference throughput improvement** due to reduced memory bandwidth pressure.

**Why bandwidth, not compute, is the bottleneck:** Modern GPUs (H100) have ~3.35 PFLOPS of BF16 compute but only ~3.35 TB/s of HBM bandwidth. For decoding, the arithmetic intensity (FLOPs per byte moved) is extremely low — you move gigabytes of KV cache to compute a handful of dot products. The GPU is bandwidth-bound, not compute-bound.

### 4.2 Grouped Query Attention (GQA)

**Ainslie et al., 2023.** Generalizes the spectrum: $g$ KV heads, with $h/g$ query heads sharing each KV head.

- $g = h$: Full MHA
- $g = 1$: MQA  
- $1 < g < h$: GQA (the sweet spot)

Llama-2 70B uses $h=64$ query heads, $g=8$ KV heads → 8 query heads share 1 KV pair. **KV cache is 8× smaller than MHA** while retaining ~98% of the quality.

Llama-3 and Mistral also use GQA as the default. It has become the de facto standard for production-scale models.

**Mathematical framing:** GQA is equivalent to MHA where the $W^K$ and $W^V$ matrices are block-diagonal with tied blocks. The expressivity reduction is bounded and empirically benign for most downstream tasks.

---

In [ ]:
# 5. Flash Attention: Hardware-Aware Algorithm Design

## 5. Flash Attention: Hardware-Aware Algorithm Design

### 5.1 The Problem: Memory Hierarchy Mismatch

Standard attention materializes the full $n \times n$ score matrix in HBM:

1. Compute $S = QK^T / \sqrt{d_k}$ → write $S$ to HBM ($O(n^2)$ memory)
2. Read $S$ from HBM, compute $P = \text{softmax}(S)$ → write $P$ to HBM
3. Read $P$ and $V$ from HBM, compute $O = PV$ → write $O$ to HBM

Every step is an HBM read/write. HBM bandwidth on an H100 is ~3.35 TB/s, but the round-trip latency for large matrices is the bottleneck. GPU SRAM (L1 cache + shared memory) is ~20 MB total but has ~19 TB/s effective bandwidth — nearly **6× faster**.

For $n=8192$, $d_k=128$: the $n \times n$ matrix is $8192^2 \times 2$ bytes $\approx$ **134 MB** — far too large for SRAM.

### 5.2 The Solution: Tiling + Online Softmax

Flash Attention (**Dao et al., 2022**) uses two key techniques:

**Tiling:** Break $Q$, $K$, $V$ into blocks of size $B_r \times d_k$ and $B_c \times d_k$ that fit in SRAM. Process block by block.

**Online softmax (the mathematical trick):** Softmax requires the full row sum $\sum_j e^{s_{ij}}$ to normalize. But you can maintain a running maximum $m$ and running sum $\ell$ and update them incrementally:

For block $t$ with max score $m_t^{\text{new}} = \max(m_t^{\text{old}}, \text{rowmax}(S_t))$:

$$\ell^{\text{new}} = e^{m^{\text{old}} - m^{\text{new}}} \ell^{\text{old}} + \sum_j e^{s_{ij} - m^{\text{new}}}$$

$$O^{\text{new}} = \frac{e^{m^{\text{old}} - m^{\text{new}}} \ell^{\text{old}} \cdot O^{\text{old}} + e^{S_t - m^{\text{new}}} V_t}{\ell^{\text{new}}}$$

This is numerically equivalent to the standard softmax but never needs the full row simultaneously.

**Kernel fusion:** The entire attention computation (QK multiply, mask, softmax, AV multiply) is fused into a single CUDA kernel, eliminating intermediate HBM round-trips.

**Result:**
- Memory: $O(n^2)$ → $O(n)$ (no full matrix materialized)
- Speed: **2-4× faster** than standard attention (even with ~same FLOPs), because GPU cores stop starving for data
- Enables context lengths of 100K+ tokens that were previously impossible

### 5.3 Flash Attention 2 & 3

**FA2** added better parallelism over the sequence length dimension and reduced non-matmul FLOPs. **FA3** (2024) targets H100's new features: asynchronous memory copies and FP8 compute, squeezing even more out of the hardware.

---

In [ ]:
# 6. Sliding Window Attention

## 6. Sliding Window Attention

### 6.1 Architecture

Used in Mistral 7B. Each token attends only to the $W$ most recent tokens (window size $W = 4096$ in Mistral). The attention matrix is banded rather than lower-triangular.

**Complexity:** $O(n \cdot W)$ instead of $O(n^2)$.

### 6.2 The Receptive Field Argument

The concern: does the model "forget" the beginning of a 100K document?

With $L$ layers each with window $W$, the **effective receptive field** at layer $L$ is $L \times W$. For Mistral with $L=32$, $W=4096$: effective receptive field = $131,072$ tokens. The information from early tokens doesn't disappear — it propagates forward through intermediate representations at each layer.

This is analogous to dilated convolutions in CNNs: local operations at each layer, but exponentially growing receptive fields with depth.

**The limitation:** Unlike full attention where there's a *direct* gradient path between any two tokens, sliding window creates an *indirect* path through intermediate layers. Fine-grained token-to-token interaction across long distances requires multiple hops, which can degrade precision for certain retrieval tasks. This is the origin of the "lost in the middle" problem for very long contexts.

---

In [ ]:
# 7. Positional Encodings

## 7. Positional Encodings

### 7.1 The Permutation Invariance Problem

Raw transformer attention computes $\text{softmax}(QK^T/\sqrt{d_k})V$. This operation is **permutation equivariant**: shuffle the input tokens, and the output is shuffled in the same way. The model has no intrinsic notion of "token 5 comes before token 6."

We must inject positional information explicitly.

---

### 7.2 Absolute Sinusoidal (Original Transformer)

$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right), \quad PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

Different dimensions encode position at different frequencies (like a Fourier decomposition). The dot product between position embeddings at positions $p$ and $q$ depends only on $p - q$, giving some relative sensitivity.

**Failure mode:** At inference on sequences longer than the training length, the sinusoidal values extrapolate to positions the model was never trained on. Embeddings degrade. Performance collapses.

---

### 7.3 Learned Absolute Embeddings

Replace the formula with a learned lookup table $E \in \mathbb{R}^{n_{max} \times d_{model}}$. Works well within training distribution. **Same failure mode:** position indices $> n_{max}$ have no learned embedding.

---

### 7.4 RoPE (Rotary Positional Embedding)

**Su et al., 2021.** Used in Llama, PaLM, Falcon, and most modern models.

**Core idea:** Rather than adding a positional vector to the embedding, apply a **rotation** to the query and key vectors based on position. For a pair of dimensions $(2i, 2i+1)$ at position $m$:

$$\begin{pmatrix} q_{2i}^m \\ q_{2i+1}^m \end{pmatrix} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} q_{2i} \\ q_{2i+1} \end{pmatrix}$$

where $\theta_i = 10000^{-2i/d_k}$.

**The elegant property:** The dot product between rotated $q^m$ and $k^n$ depends only on the relative position $m - n$:

$$\langle R_m q, R_n k \rangle = \langle q, R_{n-m} k \rangle$$

The absolute positions cancel. The model learns purely relative relationships. It generalizes well beyond training length because the rotation angles are defined analytically (not via lookup table).

**RoPE scaling for long context:** To extend context beyond training length, you can scale the base $10000$ upward (e.g., to $500000$ in Llama-3), effectively compressing the rotation frequencies to allow longer relative positions to remain distinguishable.

---

### 7.5 ALiBi (Attention with Linear Biases)

**Press et al., 2021.** Does not modify embeddings at all. Instead, before softmax, subtracts a head-specific linear penalty proportional to token distance:

$$\text{score}_{ij} = q_i \cdot k_j / \sqrt{d_k} - m_h \cdot (i - j)$$

where $m_h$ is a head-specific slope (set geometrically across heads). Closer tokens pay less penalty; distant tokens pay more. This **hard-wires a local inductive bias** directly into the attention scores.

**Extrapolation advantage:** Because the bias is linear and algebraically simple, the model extrapolates to unseen sequence lengths smoothly. Used in BLOOM and MPT models.

**Tradeoff vs. RoPE:** ALiBi's linear decay may be too aggressive for tasks requiring long-range reasoning (e.g., retrieval over long documents). RoPE's decay is softer and more nuanced. RoPE has largely won the practical competition.

---

In [ ]:
# 8. Normalization: Pre-Norm, Post-Norm, RMSNorm

## 8. Normalization: Pre-Norm, Post-Norm, RMSNorm

### 8.1 Layer Norm

$$\text{LayerNorm}(x) = \frac{x - \mu}{\sigma + \epsilon} \cdot \gamma + \beta$$

where $\mu$ and $\sigma$ are computed per token across the feature dimension.

**Post-Norm (original transformer):** Applied after the residual: $x \leftarrow \text{LN}(x + \text{Sublayer}(x))$. The gradient must flow through the normalization, which rescales it. Deep networks (>12 layers) have unstable training with post-norm.

**Pre-Norm (modern default):** Applied before: $x \leftarrow x + \text{Sublayer}(\text{LN}(x))$. The residual stream $x$ passes through the network **unnormalized and uninterrupted**. Gradients flow back through the highway with zero impedance. Llama, GPT-NeoX, and virtually all modern models use pre-norm.

---

### 8.2 RMSNorm

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot \gamma, \quad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^d x_i^2}$$

Drops the mean-centering step from LayerNorm. **Justification:** Zhang & Sennrich (2019) showed that the re-centering step contributes negligible representational benefit but requires an extra pass over the data (a synchronization point on GPU). RMSNorm achieves **10–30% throughput improvement** with no measurable quality loss. Used in Llama and most recent architectures.

---